In [1]:
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')

gat_config = GATConfig.gpt_size("small", [BOS_TOKEN_ID+1, 128])
model = GAT(gat_config)

# ---- load ckpt ----
def local_load_ckpt(model, ckpt_path):
    """load compiled ckpt locally"""
    model_ckpt = torch.load(ckpt_path, map_location="cpu")['model']
    clean_state_dict = {k.replace("_orig_mod.", ""): v for k, v in model_ckpt.items()}
    model.load_state_dict(clean_state_dict)
    return model

ckpt_path = "ckpt/ts-k4-v128.pt"
model = local_load_ckpt(model, ckpt_path)
K = 4

# model = model.to("cuda")
# model = torch.compile(model)

In [34]:
from data.tinystory_local import TinyStoriesDataLoader, collect_rollout_statistics, AbstractionStatistics
from data.tinystory_local import visualize_dynamics
import tiktoken 

# TinyStories Dataset + GPT2 tokenizer
# -------------------------------------
num_stories = 100
max_len = 64
doc_len = max_len  + (max_len - 1) // K # <-- doc len contains abstract tokens
loader = TinyStoriesDataLoader(num_stories=num_stories, max_len=max_len, chunk_size=K, device='cpu', split="train")

batch_size = 8
memory_span = 1792
attn_blocksize = 1792
max_iterations = 2

# ---- stat collection ---
abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# --- tokenizer --- 
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>']

Loading 100 stories from TinyStories train...
Loaded 100 stories, 6400 tokens total, 0.02 MB
Collected 1392 unique 4-chunks


In [ ]:
# Idea #0. Get it to talk shit
# Idea #1. Check for 'search advantage' scaling
# Idea #2. Design proper Catastrophic forgetting probing experiment

# loader.stories

In [ ]:
# (I). Generate with SoRL trained on TinyStories Dataset
# -------------------------------------------------
from sorl.neo_utils import generate
from data.tinystory_local import visualize_interleaved_alignment    

min_temperature = 0.0
tokens, doc_ids = loader.get_batch(1)

# idx = tokens[:, :15].clone()
idx = torch.tensor(enc.encode("Chrismas is coming soon, ")).unsqueeze(0)

img_frames = []
for i in range(120): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    img = visualize_interleaved_alignment(idx, model, enc, K=K, max_chunks=8)
    img_frames.append(img)

# ---- save to gif ----
if len(img_frames) > 0:
    img_frames[0].save(
        'tiny-tiny-stories-generation.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds sper frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

Saved GIF with 120 frames


In [24]:
# (II). Search Advantage Scaling checks



In [35]:
from sorl.forget import compute_abs_stats

n = 10
temperature = torch.tensor([0.0] + [5.0] * (n - 1))

# ---- statistics on info gain reward ---
i = 0
batch_size = 2
ig = []
base_traj_loss = []
cond_traj_loss = []
greedy_adv = []

from tqdm import tqdm

with tqdm(total=len(loader.stories), desc="Processing batches") as pbar:
    while i < len(loader.stories): 
        batch_indices = torch.arange(i, min(i + batch_size, len(loader.stories)))
        loader.get_specific(batch_indices) 
        i += batch_size
        tokens, doc_ids = loader.get_specific(batch_indices) 
        traj_loss, abs_logits, abs_tokens, doc_rel_info_gain, rel_info_gain, g_adv, base_ppt = compute_abs_stats(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperature,
                                                                                            truncate_seq_len=False)
        # abs_stats.update(abs_logits, traj_loss, doc_rel_info_gain, abs_tokens, doc_ids)
        ig.append(rel_info_gain)
        cond_traj_loss.append(traj_loss.mean().item())
        base_traj_loss.append(base_ppt.mean().item())
        greedy_adv.append(g_adv.mean().item())
        pbar.update(len(batch_indices))
    
print("base traj loss: ", torch.tensor(base_traj_loss).mean().item())
print("cond traj loss: ", torch.tensor(cond_traj_loss).mean().item())
print("greedy adv: ", torch.tensor(greedy_adv).mean().item())
print("info gain: ", torch.tensor(ig).mean().item())

Processing batches: 100%|██████████| 100/100 [01:17<00:00,  1.28it/s]

base traj loss:  1.2006586790084839
cond traj loss:  0.9909495711326599
greedy adv:  0.17727334797382355
info gain:  0.016000743955373764


In [ ]:
# ----- (prefix truncation free ver.) ----- 
from src.utils import * 
from src.utils import _load_data_shard

filename_pattern = "data/fineweb10B/fineweb_train_*.bin"
sequence_length = 1024 
world_size = 1
rank = 0

def distributed_data_generator_sorl(filename_pattern: str, sequence_length: int, rank : int, world_size : int):
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    assert sequence_length % world_size == 0
    local_seq_len = sequence_length // world_size
    file_iter = itertools.cycle(files) # iter(files) instead if you want to do 1-epoch training
    tokens, pos = _load_data_shard(next(file_iter)), 0
    while True:
        if pos + sequence_length + 1 >= len(tokens):
            tokens, pos = _load_data_shard(next(file_iter)), 0
        buf = tokens[pos + rank * local_seq_len:][:local_seq_len + 1]        
        idx = buf[None, :-1].to(device="cuda", dtype=torch.int32, non_blocking=True)
        pos += sequence_length
        yield idx

def distributed_data_generator_sorl_v2(filename_pattern: str, sequence_length: int, rank: int, world_size: int):
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    assert sequence_length % world_size == 0
    local_len = sequence_length // world_size
    file_iter = itertools.cycle(files) # iter(files) instead if you want to do 1-epoch training
    tokens, pos = _load_data_shard(next(file_iter)), 0

    while True:
        start = pos + rank * local_len

        matches = (tokens[start:] == 50256).nonzero() if start < len(tokens) else []
        if len(matches) == 0 or start + matches[0] + local_len + 1 > len(tokens):
            tokens, pos = _load_data_shard(next(files)), 0
            continue

        real_start = start + matches[0].item()
        buf = tokens[real_start : real_start + local_len + 1]
        idx = buf[None, :-1].to(device="cuda", dtype=torch.int32, non_blocking=True)
        pos += sequence_length
        yield idx

In [ ]:
# ----- rebuilt of evaluate function -----
from sorl.forget import * 

def compute_abs_stats(tokens, model, n, K, max_iterations, memory_span, attn_blocksize, temperature, truncate_seq_len):
        
    search_data, search_ppt, abs_logits = sorl_rollout_v3(tokens, model, n=n, K=K, 
                                                            max_iterations=max_iterations,
                                                            memory_span=memory_span,
                                                            attn_blocksize=attn_blocksize,
                                                            temperature=temperature,
                                                            truncate_seq_len=truncate_seq_len)
    search_ppt = search_ppt.reshape(search_data.shape[0], -1)

    # Get valid positions
    bos_pos_mask = torch.logical_and(
        search_data[:, :-1] != BOS_TOKEN_ID, 
        search_data[:, 1:] != BOS_TOKEN_ID
    ).float()

    traj_mask = (search_data[:, 1:] < model.vocab_sizes[0]).float()

    # --- greedy rollout's advantage ---
    valid_traj_mask = bos_pos_mask * traj_mask
    raw_ppt_adv = (search_ppt[1:].mean(dim=0) - search_ppt[0]) / (search_ppt[1:].mean(dim=0) + 1e-8)
    greedy_adv = (raw_ppt_adv * valid_traj_mask[0]).sum() / valid_traj_mask[0].sum().clamp(min=1)

    # --- losses ---
    greedy_ppt = search_ppt[0]
    abs_mask = 1 - traj_mask[0]

    valid_traj = valid_traj_mask[0]
    traj_ppt = search_ppt[0] * valid_traj
    doc_idx = (search_data == BOS_TOKEN_ID).cumsum(dim=1)
    doc_idx = doc_idx - doc_idx.min(dim=1, keepdim=True).values # idx starts from 0  

    # --- base traj loss ---
    base_traj_ppt, _  = model.forward(tokens, memory_span, attn_blocksize)
    levels = (search_data >= model.vocab_sizes[0]).long()
    best_data, best_ppt, _, info_gain_reward = select_best_info_gain(tokens, base_traj_ppt, search_data, search_ppt, levels)
    best_traj_ppt = best_ppt[best_data[0, 1:] < model.vocab_sizes[0]]
    info_gain = (base_traj_ppt[..., :best_traj_ppt.shape[-1]] - best_traj_ppt) # traj ppt might be truncated    
    
    rel_info_gain = info_gain.mean() / base_traj_ppt[..., :best_traj_ppt.shape[-1]].mean() # focus on hard case

    traj_mask = best_data[0, 1:] < model.vocab_sizes[0]
    traj_doc_idx = doc_idx[0, 1:][traj_mask]
    doc_info_gain = avg_ppt_per_sample(info_gain.unsqueeze(0), traj_doc_idx.unsqueeze(0)).squeeze(0)
    doc_base_traj_ppt = avg_ppt_per_sample(base_traj_ppt.unsqueeze(0), traj_doc_idx.unsqueeze(0)).squeeze(0)
    doc_rel_info_gain = doc_info_gain / doc_base_traj_ppt

    return greedy_ppt, doc_rel_info_gain, rel_info_gain, greedy_adv, base_traj_ppt.mean() 


In [63]:
def _load_data_shard(file: Path):
    header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
    assert header[0] == 20240520, "magic number mismatch in the data .bin file"
    assert header[1] == 1, "unsupported version"
    num_tokens = int(header[2]) # number of tokens (claimed)
    with file.open("rb", buffering=0) as f:
        tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # avoid pin_memory copy by @YouJiacheng
        f.seek(256 * 4)
        nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
        assert nbytes == 2 * num_tokens, "number of tokens read does not match header"
    return tokens

# ----- necessary sacrifice here ----- 
# -> I see why my brain hated it before ...
local_seq_len = 1024
pos = 333 
rank = 0

files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
tokens, pos = _load_data_shard(files[0]), 0 

buf_truncate = tokens[pos + rank * local_seq_len:]

# Find the index of the first BOS_TOKEN_ID (50256)

first_bos_idx = (buf_truncate == 50256).nonzero(as_tuple=False)
if len(first_bos_idx) > 0:
    first_bos_idx = first_bos_idx[0].item()
    suffix = buf_truncate[first_bos_idx:]
else:
    suffix = buf_truncate # fallback if BOS not found


In [60]:
buf_truncate == BOS_TOKEN_ID

tensor([False, False, False,  ..., False, False, False])

In [61]:
BOS_TOKEN_ID

50256

In [ ]:
# first 100 stories in training set, truncated to 64 tokens for each data
# n=2, info gain: 1.39% | greedy adv: 17.3%
# n=5, info gain: 1.54% | greedy adv: 17.7%
# n=10, info gain: 1.57% | greedy adv: 17.6% 

# first 100 stories in validation set, truncated to 64 tokens for each data
# n=2, info gain: 2.2% | greedy adv: 38.2%
# n=5, info gain: 2.2% | greedy adv: 38.4%
# n=10, info gain: 2.2% | greedy adv: 38.4%

# There is an issue with training logs
# 1. truncated prefix, the data generator randomly truncates prefix in a batch (to keep constant length for each batch)

# 1. Directly hurts p(a | s) by removing prefix conditioning, making generated abstraction less meaningful
#    This further leads to worse p(s | a), info gain and search advantage, and unstable training signal

# We got 16 * 1024 length seq for each batch, if the avg length of TinyStories is 2048, then training set has 
# an avg. percentage of 1/8 of sequence having its prefix truncated, this creates a heavy bias for training set
# this also explains why I observe a much higher greedy adv & info gain in validation set, as the weird prefix 
# removal bias is not injected into validation set

In [ ]:
from sorl.forget import collect_forget_data, plot_normalized_correlation_lines, train_forget_vec
from tqdm import tqdm as tqdm

# ---- Memory Interference Experiment ---

abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

abs_stats_post = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# ---- Compute 'Forget Matrix' --- 
forget_mat = torch.zeros(num_stories, num_stories, device=model.device)
num_steps = 40
for train_idx in tqdm(range(num_stories)): 
    forget_vec = train_forget_vec(train_idx, loader, model, abs_stats, abs_stats_post, optimizer, num_steps,
                                max_iterations, memory_span, attn_blocksize, temperature, K, r_min, reward_mode, loss_fn, alpha_abs, alpha_soft_zipf, alpha_topo,
                                ckpt_path="sorl_tinystories.pt")
    forget_mat[train_idx] = forget_vec 

# torch.save(forget_mat, "forget_mat.pt") # save it just in case

# # ---- Visualize 'Forget Matrix' --- 
# correlation_data, ham_corrs = collect_forget_data(forget_mat, abs_stats)

# plot_normalized_correlation_lines(
#     correlation_data, 
#     ham_corrs,
#     xlabel="Abstraction Edit Distance", 
#     ylabel="Forgetting (Δ Perplexity)",
#     title="Distant Abstractions → Less Forgetting (TinyStories)"
# )

In [ ]:
# Exp #1. 
# 'self-organization' of abstraction representation --> will SoRL tries to pull mixing memory apart? 
# Exp #2. 
# setting up tiny memory dataset training pipeline 
# Question #1. 
# What if we use all-rollout SoRL with utility advantage + KL reg terms? 


In [ ]:
from data.tinystory_local import *
from sklearn.decomposition import PCA

cs_sim = abs_stats.compute_cross_doc_logit_sim()
# cs_sim = abs_stats.compute_cross_doc_hamming()
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(cs_sim.cpu().numpy())

# Now use in 3D visualization
train_idx = 0
perplexity = forget_mat[train_idx]
img = visualize_forget_terrain(coords_2d, perplexity, trained_idx=train_idx, step=step)
# img = visualize_perplexity_terrain(coords_2d, perplexity, trained_idx=train_idx, step=step)

In [4]:
# Hypothesis #2. 
# ------------------------------------------------------------
# forget(i | j) is proportional to 1 / d(a_i, a_j)
# dis-similar concept is less likely to be overwritten by each other
# similar concept is more likely to be overwritten by each other
# the emerged abstraction system from SoRL can describe such similarity via d(a_i, a_j)
# ------------------------------------------------------------

# Experiment 
# (a). Train till emergence
# (b). Train with specific order. 
# (c). Record 'forgetting matrix'

import numpy as np
np.array(record['topo_loss'][:5]).mean(), np.array(record['topo_loss'][-5:]).mean()

(-0.7774280548095703, -0.7865824460983276)

In [4]:
if len(img_frames) > 0:
    img_frames[0].save(
        'tinystories_dynamics (select-best per abs SoRL + 1.0 topo reg + 1.0 bigram zipf reg + utility reward scaling.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

Saved GIF with 200 frames


In [ ]:
# Generate with SoRL trained on TinyStories Dataset
# -------------------------------------------------
from sorl.neo_utils import generate
from data.tinystory_local import visualize_interleaved_alignment    

min_temperature = 0.0
tokens, doc_ids = loader.get_batch(1)

idx = tokens[:, :15].clone()

img_frames = []
for i in range(30): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    img = visualize_interleaved_alignment(idx, model, enc, K=K, max_chunks=8)
    img_frames.append(img)

# ---- save to gif ----
if len(img_frames) > 0:
    img_frames[0].save(
        'tiny-tiny-stories-generation.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

In [ ]:
# Request #1. 
# -> Full scale experiment on TinyStories & FineWeb

In [ ]:
# marg cond w = 1.0 
# traj_loss: 0.85 | abs_loss: 0.04 | search adv: 21.01% | vocab util: 93.75%  | marg_ent: 2.67 | cond_ent: 0.05 | avg cos sim: 0.44

# soft bigram zipf kl w = 1.0 
# traj_loss: 0.89 | abs_loss: 0.53 | search adv: 25.84% | vocab util: 68.75%  | kl_soft_zipf: 0.24 | avg cos sim: 0.47
# -> visually I observe much less 'repetitions'

# soft bigram zipf kl w = 1.0 & utility reward scaling r = max(p(s|a)/p(s), 1.0)
# traj_loss: 0.79 | abs_loss: 0.40 | search adv: 30.13% | vocab util: 62.50%  | kl_soft_zipf: 0.17 | avg cos sim: 0.44 | topo sim: 0.77 

# soft bigram zip w=1.0 & utility reward scaling & topo reg w=1.0
# traj loss: 1.50 | search adv: 20% | avg cos sim: 0.24 | topo sim: 0.53
# => degrades utility, no go

